# Notebook 10 — East Asian Precipitation Forecast Skill
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

Evaluate how well each of the three 2-D representations forecasts East Asian
summer precipitation at lead times τ = 0, +5, +10 days.

**Representations (2-D scalar: θ = atan2(z₂, z₁))**
| Name | Source | Notes |
|------|--------|-------|
| `idx` | BSISO index PC1/PC2 | from BSISO.INDEX.NORM.LY.data |
| `sup` | Supervised 2D encoder (128-layer, nb 07c) | `lee_2d_no_l2/embeddings.npy` |
| `ssl` | SSL temporal 2D encoder (32-layer, nb 08) | `lee_2d_ssl/embeddings.npy` |

**Skill metric:** Anomaly Correlation Coefficient (ACC) between predicted and observed precipitation anomaly  
**Predictor:** cos(θ) and sin(θ) → 2-feature linear regression per grid point  
**Target:** Lee-preprocessed daily `tp` anomaly at lead τ  
**Evaluation:** leave-one-year-out cross-validation (same split as nb 07c / 08)

**Outputs:**
```
results/precip_forecast/
  spatial_acc_maps.png        — 3×3 panel: 3 repr × 3 lead times
  ea_headline_scores.png      — bar chart, EA area-avg ACC vs lead time
  skill_table.csv             — numeric ACC table
  precip_forecast_report.txt  — plain-text summary
```

## Cell 1 — Mount Drive & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
RAW_DIR       = f'{PROJECT_DIR}/data/raw'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
RESULTS_DIR   = f'{PROJECT_DIR}/results'
OUT_DIR       = f'{RESULTS_DIR}/precip_forecast'
os.makedirs(OUT_DIR, exist_ok=True)

# Embedding files (both use same MJJAS Lee-preprocessed data)
SUP_EMB_FILE    = f'{RESULTS_DIR}/lee_2d_no_l2/embeddings.npy'     # 128-layer supervised
SSL_EMB_FILE    = f'{RESULTS_DIR}/lee_2d_ssl/embeddings.npy'       # 32-layer SSL
LABELS_SUP_FILE = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee.csv'  # aligned to sup
LABELS_SSL_FILE = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee_lp25.csv'  # aligned to ssl
BSISO_RAW_FILE  = f'{RAW_DIR}/BSISO.INDEX.NORM.LY.data'
PRECIP_FILE     = f'{RAW_DIR}/precip_MJJAS_1979_2023.nc'

for f in [SUP_EMB_FILE, SSL_EMB_FILE, LABELS_SUP_FILE, LABELS_SSL_FILE,
          BSISO_RAW_FILE, PRECIP_FILE]:
    tag = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'[{tag}] {os.path.basename(f)}')

## Cell 2 — Load Embeddings & Build θ Arrays

In [ ]:
# ── supervised embeddings ──────────────────────────────────────────────────
emb_sup  = np.load(SUP_EMB_FILE)   # (N_sup, 2)
df_sup   = pd.read_csv(LABELS_SUP_FILE, parse_dates=['date'])
df_sup['date'] = df_sup['date'].dt.normalize()
assert len(emb_sup) == len(df_sup), 'sup embedding/label length mismatch'

theta_sup  = np.arctan2(emb_sup[:, 1], emb_sup[:, 0])   # (N_sup,)
dates_sup  = pd.DatetimeIndex(df_sup['date'].values)

# ── SSL embeddings ─────────────────────────────────────────────────────────
emb_ssl  = np.load(SSL_EMB_FILE)   # (N_ssl, 2)
df_ssl   = pd.read_csv(LABELS_SSL_FILE, parse_dates=['date'])
df_ssl['date'] = df_ssl['date'].dt.normalize()
assert len(emb_ssl) == len(df_ssl), 'ssl embedding/label length mismatch'

# Negate z₂ to flip the SSL traversal direction from counter-clockwise to
# clockwise, matching the idx/sup convention.  The SSL encoder was randomly
# initialised (no torch.manual_seed), so its 2D output traverses BSISO phases
# in reverse order; reflecting across the z₁ axis corrects this post-hoc,
# giving ρ_c(idx,ssl) > 0 and ρ_c(sup,ssl) > 0 without retraining.
theta_ssl  = np.arctan2(-emb_ssl[:, 1], emb_ssl[:, 0])  # (N_ssl,)
dates_ssl  = pd.DatetimeIndex(df_ssl['date'].values)

# ── BSISO index angle ──────────────────────────────────────────────────────
rows = []
with open(BSISO_RAW_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line[0].isalpha() or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) >= 4:
            rows.append(parts)
col_names = ['year','doy','pc1_bsiso1','pc2_bsiso1',
             'pc1_bsiso2','pc2_bsiso2','bsiso1_amp','bsiso2_amp']
df_bsiso = pd.DataFrame(rows, columns=col_names[:len(rows[0])])
for c in df_bsiso.columns:
    df_bsiso[c] = pd.to_numeric(df_bsiso[c], errors='coerce')
df_bsiso['date'] = pd.to_datetime(
    df_bsiso['year'].astype(int).astype(str) +
    df_bsiso['doy'].astype(int).astype(str).str.zfill(3),
    format='%Y%j').dt.normalize()

# Merge onto sup dates to get idx angle for every sup-day
df_idx = df_sup[['date']].merge(df_bsiso[['date','pc1_bsiso1','pc2_bsiso1']],
                                on='date', how='left')
pc1_idx = df_idx['pc1_bsiso1'].values.astype(float)
pc2_idx = df_idx['pc2_bsiso1'].values.astype(float)
theta_idx = np.arctan2(pc2_idx, pc1_idx)   # (N_sup,)
dates_idx  = dates_sup                      # same set of dates

n_nan = int(np.isnan(theta_idx).sum())
if n_nan > 0:
    print(f'WARNING: {n_nan} NaN in theta_idx — dates not found in BSISO file')

print(f'sup  N={len(theta_sup)}, ssl  N={len(theta_ssl)}, idx  N={len(theta_idx)}')
print('Embeddings loaded.')

## Cell 3 — Load & Preprocess Precipitation

In [ ]:
import xarray as xr

# ── load raw precip ────────────────────────────────────────────────────────
ds = xr.open_dataset(PRECIP_FILE)
time_dim = 'valid_time' if 'valid_time' in ds.dims else 'time'
tp_var   = 'tp' if 'tp' in ds.data_vars else list(ds.data_vars)[0]

times_raw = pd.DatetimeIndex(ds[time_dim].values).normalize()
tp_raw    = np.clip(ds[tp_var].values.astype(np.float32), 0, None)  # (T, lat, lon)
lats      = ds.latitude.values
lons      = ds.longitude.values
ds.close()

T, nlat, nlon = tp_raw.shape
print(f'Raw tp: {T} days × {nlat} lat × {nlon} lon')
print(f'Period: {str(times_raw[0])[:10]} → {str(times_raw[-1])[:10]}')

# ── Lee et al. preprocessing on tp ────────────────────────────────────────
# Step 1: remove annual cycle via 3-harmonic Fourier fit on DOY climatology
# Step 2: remove preceding 120-day running mean (interannual)
# Step 3: normalize by area-averaged temporal std (one scalar)

CLIM_START, CLIM_END = 1981, 2010

# Build DOY climatology (1-366) on clim period
clim_mask = np.isin(times_raw.year, range(CLIM_START, CLIM_END + 1))
times_clim = times_raw[clim_mask]
tp_clim    = tp_raw[clim_mask]

doys     = np.arange(1, 367)
n_doy    = len(doys)
clim_map = np.zeros((n_doy, nlat, nlon), dtype=np.float32)  # raw clim per DOY
counts   = np.zeros(n_doy, dtype=int)
for t_idx, d in enumerate(times_clim.dayofyear):
    clim_map[d - 1] += tp_clim[t_idx]
    counts[d - 1]   += 1
for d in range(n_doy):
    if counts[d] > 0:
        clim_map[d] /= counts[d]

# Fit 3 harmonics to clim_map (broadcast over space)
t_rad = 2 * np.pi * doys / 365.25
basis = np.column_stack([np.ones(n_doy)] +
                        [f(k * t_rad) for k in range(1, 4)
                         for f in (np.cos, np.sin)])  # (366, 7)
clim_flat = clim_map.reshape(n_doy, -1)              # (366, nlat*nlon)
coef, *_ = np.linalg.lstsq(basis, clim_flat, rcond=None)
clim_fit  = (basis @ coef).reshape(n_doy, nlat, nlon).astype(np.float32)

# Step 1: subtract fitted annual cycle
doy_all     = times_raw.dayofyear
tp_anom     = tp_raw - clim_fit[doy_all - 1]        # (T, nlat, nlon)

# Step 2: subtract preceding 120-day running mean
W = 120
tp_rm = np.full_like(tp_anom, np.nan)
for i in range(T):
    start = max(0, i - W)
    tp_rm[i] = tp_anom[start:i].mean(axis=0) if i > 0 else 0.0
tp_anom2 = tp_anom - tp_rm

# Step 3: normalize by area-averaged temporal std
area_std = tp_anom2.std(axis=0).mean()   # scalar
tp_norm  = (tp_anom2 / area_std).astype(np.float32)

# Build fast date → row index lookup
date_to_tp_idx = {d: i for i, d in enumerate(times_raw)}

print(f'tp_norm shape: {tp_norm.shape},  area_std={area_std:.6f} m')
print('Precipitation preprocessing done.')

## Cell 4 — Linear Regression Skill (ACC, leave-one-year-out)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

LEAD_TIMES = [0, 5, 10]   # τ in days (predictor date → target date + τ)

def build_XY(theta, dates, lead, date_to_tp_idx, tp_norm):
    """Build predictor matrix X and target matrix Y for a given lead time."""
    X_rows, Y_rows, keep_years = [], [], []
    for i, d in enumerate(dates):
        if np.isnan(theta[i]):
            continue
        d_target = d + pd.Timedelta(days=int(lead))
        if d_target not in date_to_tp_idx:
            continue
        if d_target.year != d.year:     # same-year constraint
            continue
        tp_row = tp_norm[date_to_tp_idx[d_target]]   # (nlat, nlon)
        X_rows.append([np.cos(theta[i]), np.sin(theta[i])])
        Y_rows.append(tp_row.ravel())
        keep_years.append(d.year)
    X = np.array(X_rows, dtype=np.float32)           # (N, 2)
    Y = np.array(Y_rows, dtype=np.float32)           # (N, nlat*nlon)
    years = np.array(keep_years)
    return X, Y, years


def acc_loyo(X, Y, years):
    """
    Leave-one-year-out cross-validated ACC at every grid point.
    Returns acc_map: (nlat*nlon,) array in [-1,1].
    """
    unique_years = np.unique(years)
    Y_pred_all = np.zeros_like(Y)
    for yr in unique_years:
        test_mask  = years == yr
        train_mask = ~test_mask
        if train_mask.sum() < 10:
            continue
        sc = StandardScaler()
        X_tr = sc.fit_transform(X[train_mask])
        X_te = sc.transform(X[test_mask])
        model = Ridge(alpha=1.0)
        model.fit(X_tr, Y[train_mask])
        Y_pred_all[test_mask] = model.predict(X_te)

    # ACC = Pearson r between Y_pred and Y_obs across time, per grid point
    Y_c    = Y      - Y.mean(axis=0)
    Yp_c   = Y_pred_all - Y_pred_all.mean(axis=0)
    num    = (Y_c * Yp_c).mean(axis=0)
    denom  = np.sqrt((Y_c**2).mean(axis=0) * (Yp_c**2).mean(axis=0))
    acc    = np.where(denom > 1e-10, num / denom, np.nan)
    return acc


reprs = [
    ('idx', theta_idx, dates_idx),
    ('sup', theta_sup, dates_sup),
    ('ssl', theta_ssl, dates_ssl),
]

# results[repr_name][lead] = acc_map (nlat, nlon)
results = {name: {} for name, *_ in reprs}

for name, theta, dates in reprs:
    for lead in LEAD_TIMES:
        print(f'Computing {name}, lead={lead:+d} ...', end=' ', flush=True)
        X, Y, years = build_XY(theta, dates, lead, date_to_tp_idx, tp_norm)
        print(f'N={len(X)}', end=' ', flush=True)
        acc_flat = acc_loyo(X, Y, years)
        results[name][lead] = acc_flat.reshape(nlat, nlon)
        ea_mask_lat = (lats >= 20) & (lats <= 45)
        ea_mask_lon = (lons >= 100) & (lons <= 145)
        ea_acc = results[name][lead][np.ix_(ea_mask_lat, ea_mask_lon)]
        print(f'EA ACC={np.nanmean(ea_acc):.3f}')

print('\nAll done.')

## Cell 5 — Spatial ACC Maps (3 repr × 3 lead times)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('ACC Skill Map — leave-one-year-out cross-validation\n'
             'Predictor: [cos θ, sin θ]  →  Lee-anomaly precipitation at lead τ',
             fontsize=13, fontweight='bold')

VMAX = 0.4
cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-VMAX, vcenter=0, vmax=VMAX)
extent = [lons.min(), lons.max(), lats.min(), lats.max()]

row_labels = [
    'BSISO Index (idx)\nθ from PC1/PC2',
    'Supervised 2D (sup)\n128-layer encoder, nb 07c',
    'SSL 2D (ssl)\n32-layer encoder, nb 08',
]

ea_lat_mask = (lats >= 20) & (lats <= 45)
ea_lon_mask = (lons >= 100) & (lons <= 145)

for row, (name, *_) in enumerate(reprs):
    for col, lead in enumerate(LEAD_TIMES):
        ax  = axes[row, col]
        acc = results[name][lead]    # (nlat, nlon)

        im = ax.imshow(acc, cmap=cmap, norm=norm,
                       origin='upper', extent=extent, aspect='auto')

        # EA box
        rect = Rectangle((100, 20), 45, 25, linewidth=1.5,
                         edgecolor='k', facecolor='none', linestyle='--')
        ax.add_patch(rect)

        ea_acc = float(np.nanmean(acc[np.ix_(ea_lat_mask, ea_lon_mask)]))
        ax.set_title(f'τ = +{lead} d  |  EA ACC = {ea_acc:.3f}', fontsize=10)

        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=9)
        if row == 2:
            ax.set_xlabel('Longitude (°E)', fontsize=9)

        plt.colorbar(im, ax=ax, shrink=0.75, label='ACC')

plt.tight_layout(rect=[0, 0, 1, 0.95])
out_maps = f'{OUT_DIR}/spatial_acc_maps.png'
plt.savefig(out_maps, dpi=150)
plt.show()
print(f'Saved: {out_maps}')

## Cell 6 — EA Headline Bar Chart & Skill Table

In [ ]:
ea_lat_mask = (lats >= 20) & (lats <= 45)
ea_lon_mask = (lons >= 100) & (lons <= 145)

# ── skill table ───────────────────────────────────────────────────────────
table = []
for name, *_ in reprs:
    for lead in LEAD_TIMES:
        acc_map = results[name][lead]
        ea_acc  = float(np.nanmean(acc_map[np.ix_(ea_lat_mask, ea_lon_mask)]))
        full_acc = float(np.nanmean(acc_map))
        table.append({'repr': name, 'lead_days': lead,
                      'ea_acc': round(ea_acc, 4),
                      'full_domain_acc': round(full_acc, 4)})

df_table = pd.DataFrame(table)
print(df_table.to_string(index=False))
df_table.to_csv(f'{OUT_DIR}/skill_table.csv', index=False)

# ── bar chart ─────────────────────────────────────────────────────────────
colors = {'idx': '#2166ac', 'sup': '#d6604d', 'ssl': '#4dac26'}
labels = {'idx': 'BSISO Index', 'sup': 'Supervised 2D', 'ssl': 'SSL 2D'}

x = np.arange(len(LEAD_TIMES))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for k, (name, *_) in enumerate(reprs):
    ea_vals = [float(np.nanmean(
        results[name][lead][np.ix_(ea_lat_mask, ea_lon_mask)]))
        for lead in LEAD_TIMES]
    ax.bar(x + (k - 1) * width, ea_vals, width,
           label=labels[name], color=colors[name], alpha=0.85)

ax.axhline(0, color='k', linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f'τ = +{l} d' for l in LEAD_TIMES])
ax.set_ylabel('Area-averaged ACC (East Asian subregion\n20–45°N, 100–145°E)')
ax.set_title('East Asian Precipitation Forecast Skill\nleave-one-year-out CV')
ax.legend()
ax.set_ylim(-0.15, 0.55)
plt.tight_layout()
out_bar = f'{OUT_DIR}/ea_headline_scores.png'
plt.savefig(out_bar, dpi=150)
plt.show()
print(f'Saved: {out_bar}')

## Cell 7 — Write Plain-Text Report

In [ ]:
import datetime

ea_lat_mask = (lats >= 20) & (lats <= 45)
ea_lon_mask = (lons >= 100) & (lons <= 145)

lines = []
lines.append('=' * 65)
lines.append('PLAN 3 — East Asian Precipitation Forecast Skill Report')
lines.append(f'Date: {datetime.date.today()}')
lines.append('=' * 65)
lines.append('')
lines.append('Method:')
lines.append('  Predictor: [cos θ, sin θ]  (2-feature linear regression)')
lines.append('  Target:    Lee-preprocessed daily tp at grid point')
lines.append('  CV:        leave-one-year-out  (45 folds, 1979–2023)')
lines.append('  Skill:     ACC = Pearson r(predicted, observed)')
lines.append('')
lines.append('EA Subregion: 20–45°N, 100–145°E  (area-averaged ACC)')
lines.append('')
lines.append(f'{"Repr":<12} {"τ=0":>8} {"τ=+5":>8} {"τ=+10":>9}')
lines.append('-' * 40)
for name, *_ in reprs:
    row_vals = []
    for lead in LEAD_TIMES:
        ea_acc = float(np.nanmean(
            results[name][lead][np.ix_(ea_lat_mask, ea_lon_mask)]))
        row_vals.append(ea_acc)
    lines.append(f'{name:<12} {row_vals[0]:>8.3f} {row_vals[1]:>8.3f} {row_vals[2]:>9.3f}')

lines.append('')
lines.append('Files produced:')
for fn in ['spatial_acc_maps.png', 'ea_headline_scores.png', 'skill_table.csv']:
    lines.append(f'  {OUT_DIR}/{fn}')
lines.append('=' * 65)

report_text = '\n'.join(lines)
print(report_text)

report_path = f'{OUT_DIR}/precip_forecast_report.txt'
with open(report_path, 'w') as fh:
    fh.write(report_text + '\n')
print(f'\nReport saved: {report_path}')

# ── also save a copy in Desktop/DDCS ───────────────────────────────────────
desktop_out = '/content/drive/MyDrive/DDCS_local/precip_forecast_report.txt'
try:
    os.makedirs(os.path.dirname(desktop_out), exist_ok=True)
    with open(desktop_out, 'w') as fh:
        fh.write(report_text + '\n')
    print(f'Also saved to: {desktop_out}')
except Exception:
    pass  # non-critical

---
## Done

Check `results/precip_forecast/` on Google Drive for:
- `spatial_acc_maps.png` — 3×3 spatial ACC panels
- `ea_headline_scores.png` — bar chart by representation and lead time
- `skill_table.csv` — numeric ACC table (also full-domain mean)
- `precip_forecast_report.txt` — plain-text summary

**Physical interpretation guide:**
- ACC > 0.2 at τ = 0 → representation encodes precipitation-relevant structure
- ACC persists at τ = +10 → representation captures propagating intraseasonal signal
- Spatial map: higher skill over Bay of Bengal / eastern China = monsoon relevant
- Comparison: if SSL matches or exceeds supervised despite no labels → SSL captures
  physically meaningful structure, not just label-guided pattern matching

---
*DDCS Project | jh9141@nyu.edu*